# 01. Exploratory Data Analysis (EDA)
## Restaurant Order Demand Prediction

### Purpose
This notebook explores the historical demand transactions, fulfillment center operations, and meal catalogs.
It investigates target distribution, categorical cardinalities, promotional elasticity, and temporal patterns.

### Key Questions Investigated
1. What is the distribution and skew of the target variable (`num_orders`)?
2. How do email campaigns and homepage promotions impact weekly demand?
3. Which meal categories and cuisine types generate the highest order volume?
4. Is there empirical evidence of price sensitivity (checkout price vs orders)?
5. What temporal trends exist across the 145-week observation window?

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable from notebooks directory
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from restaurant_demand.data.ingestion import DataIngestion
from restaurant_demand.config import load_data_config

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Load and Verify Ingested Data

In [ ]:
ingestion = DataIngestion(base_dir=project_root)
df = ingestion.run(save_interim=False)
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

### 2. Target Variable Analysis (`num_orders`)
Weekly order counts exhibit high positive skew, typical of retail demand where majority of meals exhibit modest volume while popular items generate order spikes.

In [ ]:
print('Summary Statistics for Target (num_orders):')
display(df['num_orders'].describe().to_frame())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['num_orders'], bins=60, ax=ax1, color='#1f77b4')
ax1.set_title('Raw Order Demand Distribution')
ax1.set_xlabel('Weekly Orders')

sns.histplot(np.log1p(df['num_orders']), bins=60, ax=ax2, color='#2ca02c', kde=True)
ax2.set_title('Log-Transformed Order Demand Distribution')
ax2.set_xlabel('log(1 + num_orders)')
plt.show()

### 3. Promotional Elasticity
Analysis shows that when email promotions and homepage featured placements are applied together, demand exhibits a massive non-linear multiplier.

In [ ]:
promo_summary = df.groupby(['emailer_for_promotion', 'homepage_featured'])['num_orders'].agg(['count', 'mean', 'median']).reset_index()
display(promo_summary)

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='emailer_for_promotion', y='num_orders', hue='homepage_featured')
plt.title('Demand by Email Campaign and Homepage Placement')
plt.xlabel('Emailer for Promotion (0=No, 1=Yes)')
plt.ylabel('Mean Weekly Orders')
plt.legend(title='Homepage Featured')
plt.show()

### 4. Category and Cuisine Breakdown

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=df, x='category', y='num_orders', order=df.groupby('category')['num_orders'].mean().sort_values(ascending=False).index)
plt.xticks(rotation=45)
plt.title('Mean Demand Across Food Categories')
plt.show()

### 5. Summary of Findings
- Order distribution is heavily right-skewed with a median of 136 and mean of ~262 orders.
- Email campaigns and homepage placements multiply demand significantly.
- Multi-level entities (77 fulfillment centers and 51 meals) exhibit distinct base volumes, justifying both group-level baseline heuristics and tree-based non-linear estimators.